In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import zscore
from sklearn.ensemble import IsolationForest

In [2]:
df = pd.read_csv("data/EDA_ML/financial_data.csv")

In [3]:
df.columns

Index(['symbol', 'year_id', 'sales', 'expenses', 'operating_profit', 'opm_pct',
       'other_income', 'interest', 'depreciation', 'profit_before_tax',
       'tax_pct', 'net_profit', 'eps', 'dividend_payout_pct',
       'net_profit_margin_pct', 'expense_ratio_pct', 'interest_coverage',
       'asset_turnover', 'return_on_assets', 'equity_capital', 'reserves',
       'borrowings', 'other_liabilities', 'total_liabilities', 'fixed_assets',
       'cwip', 'investments', 'other_asset', 'total_assets', 'debt_to_equity',
       'equity_ratio', 'operating_activity', 'investing_activity',
       'financing_activity', 'net_cash_flow', 'free_cash_flow',
       'cash_conversion_ratio', 'company_name', 'sector_id', 'sub_sector',
       'company_logo', 'website', 'nse_url', 'bse_url', 'about_company',
       'sector_name', 'sector_code', 'description', 'year_label',
       'fiscal_year', 'quarter', 'is_ttm', 'is_half_year', 'sort_order'],
      dtype='str')

In [4]:
anomaly_df = df[
    ['symbol', 'year_id', 'sales', 'net_profit', 'operating_profit', 'borrowings']
].copy()

In [5]:
anomaly_df.columns

Index(['symbol', 'year_id', 'sales', 'net_profit', 'operating_profit',
       'borrowings'],
      dtype='str')

In [6]:
metrics = ['sales', 'net_profit', 'operating_profit', 'borrowings']

for col in metrics:
    anomaly_df[f'{col}_z'] = anomaly_df.groupby('symbol')[col].transform(
        lambda x: zscore(x, nan_policy='omit')
    )

In [7]:
for col in metrics:
    anomaly_df[f'{col}_anomaly_z'] = anomaly_df[f'{col}_z'].abs() > 2.5

In [8]:
anomaly_df['z_anomaly_flag'] = anomaly_df[
    [f'{col}_anomaly_z' for col in metrics]
].any(axis=1)

In [9]:
def detect_isolation(group):
    model = IsolationForest(contamination=0.05, random_state=42)
    data = group[metrics].fillna(0)
    preds = model.fit_predict(data)
    return pd.Series(preds == -1, index=group.index)

# Apply and assign back directly — symbol stays untouched
anomaly_df['iso_anomaly_flag'] = (
    anomaly_df.groupby('symbol', group_keys=False)
    .apply(detect_isolation)
)

print("Columns present:", anomaly_df.columns.tolist())

Columns present: ['symbol', 'year_id', 'sales', 'net_profit', 'operating_profit', 'borrowings', 'sales_z', 'net_profit_z', 'operating_profit_z', 'borrowings_z', 'sales_anomaly_z', 'net_profit_anomaly_z', 'operating_profit_anomaly_z', 'borrowings_anomaly_z', 'z_anomaly_flag', 'iso_anomaly_flag']


In [10]:
anomaly_df['both_flag'] = (
    anomaly_df['z_anomaly_flag'] &
    anomaly_df['iso_anomaly_flag']
)

In [11]:
print("Z anomalies:", anomaly_df['z_anomaly_flag'].sum())
print("ISO anomalies:", anomaly_df['iso_anomaly_flag'].sum())
print("Both:", anomaly_df['both_flag'].sum())

Z anomalies: 28
ISO anomalies: 91
Both: 22
